# 🩺 Diabetes Prediction using K-Nearest Neighbors (KNN)

**Author:** [Your Name]  
**Dataset:** [Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)  
**Algorithm:** K-Nearest Neighbors (KNN)  
**Tools:** Python, Pandas, Scikit-learn, Seaborn, Matplotlib

---

## 📌 Project Overview

Proyek ini bertujuan untuk **memprediksi apakah seorang pasien menderita diabetes** berdasarkan fitur-fitur medis seperti kadar glukosa, tekanan darah, indeks massa tubuh (BMI), dan lainnya.

Dataset yang digunakan berasal dari **National Institute of Diabetes and Digestive and Kidney Diseases**, terdiri dari 768 data pasien wanita berusia minimal 21 tahun dengan latar belakang suku Pima Indian.

## 🎯 Objectives
- Melakukan eksplorasi dan pembersihan data (EDA & Data Cleaning)
- Membangun model klasifikasi menggunakan algoritma KNN
- Menentukan nilai K optimal untuk performa terbaik
- Mengevaluasi model dengan confusion matrix dan accuracy score


---

## 📋 Table of Contents
1. [Import Libraries](#1-import-libraries)
2. [Load Dataset](#2-load-dataset)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis)
4. [Data Cleaning & Preprocessing](#4-data-cleaning--preprocessing)
5. [Feature Scaling](#5-feature-scaling)
6. [Model Building & K Optimization](#6-model-building--k-optimization)
7. [Model Evaluation](#7-model-evaluation)
8. [Conclusion](#8-conclusion)


---

## 1. Import Libraries

Mengimport semua library yang diperlukan untuk analisis data, visualisasi, dan pemodelan machine learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# Pengaturan tampilan
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)

print("✅ Libraries berhasil diimport")


---

## 2. Load Dataset

Memuat dataset diabetes dan menampilkan gambaran awal data.

In [ ]:
df = pd.read_csv("diabetes.csv")

print(f"📊 Shape dataset: {df.shape}")
print(f"📝 Jumlah fitur  : {df.shape[1] - 1}")
print(f"👥 Jumlah sampel : {df.shape[0]}")
df.head(10)


In [ ]:
# Informasi tipe data dan missing values
print("
📋 Informasi Dataset:")
df.info()


In [ ]:
# Statistik deskriptif
print("
📈 Statistik Deskriptif:")
df.describe().T.style.background_gradient(cmap="Blues")


---

## 3. Exploratory Data Analysis

### 3.1 Distribusi Kelas Target

Kolom  adalah variabel target:
-  = Tidak Diabetes
-  = Diabetes


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
outcome_counts = df["Outcome"].value_counts()
axes[0].bar(["Non-Diabetes (0)", "Diabetes (1)"], outcome_counts, color=["#2196F3", "#F44336"], edgecolor="white", width=0.5)
axes[0].set_title("Distribusi Kelas Target", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Jumlah")
for i, v in enumerate(outcome_counts):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(outcome_counts, labels=["Non-Diabetes", "Diabetes"], autopct="%1.1f%%",
            colors=["#2196F3", "#F44336"], startangle=90, explode=(0.05, 0.05))
axes[1].set_title("Proporsi Kelas Target", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()
print(f"
Rasio kelas: {outcome_counts[0]}/{outcome_counts[1]} (Non-Diabetes/Diabetes)")


### 3.2 Identifikasi Nilai Tidak Valid

Beberapa kolom medis tidak mungkin bernilai 0 secara biologis (misalnya , , ). Nilai 0 pada kolom tersebut kemungkinan merupakan **missing values** yang perlu ditangani.

In [ ]:
cols_with_zero = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("⚠️  Jumlah nilai 0 yang tidak valid per kolom:")
for col in cols_with_zero:
    zero_count = (df[col] == 0).sum()
    pct = zero_count / len(df) * 100
    print(f"  {col:<25}: {zero_count:>4} ({pct:.1f}%)")


### 3.3 Analisis Korelasi

In [ ]:
df_temp = df.copy()
df_temp[cols_with_zero] = df_temp[cols_with_zero].replace(0, np.nan)

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(df_temp.corr(), dtype=bool))
sns.heatmap(df_temp.corr(), annot=True, fmt=".2f", cmap="RdYlGn",
            mask=mask, linewidths=0.5, square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()


### 3.4 Pairplot Berdasarkan Outcome

In [ ]:
g = sns.pairplot(df_temp, hue="Outcome", palette={0: "#2196F3", 1: "#F44336"},
                 plot_kws={"alpha": 0.5}, diag_kind="kde")
g.fig.suptitle("Pairplot Fitur Dataset Diabetes", y=1.02, fontsize=14, fontweight="bold")
plt.show()


---

## 4. Data Cleaning & Preprocessing

Mengganti nilai 0 yang tidak valid dengan , lalu mengisi missing values menggunakan **mean imputation** per kolom.


In [ ]:
df_clean = df.copy()

# Ganti nilai 0 menjadi NaN
df_clean[cols_with_zero] = df_clean[cols_with_zero].replace(0, np.nan)

print("Missing values setelah penggantian nilai 0:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0].to_string())


In [ ]:
# Imputasi dengan mean
for col in cols_with_zero:
    df_clean[col].fillna(df_clean[col].mean(), inplace=True)

print("✅ Missing values setelah imputasi:", df_clean.isnull().sum().sum())
df_clean.describe().T.round(2)


---

## 5. Feature Scaling

KNN sangat sensitif terhadap skala fitur karena menggunakan jarak Euclidean. **StandardScaler** digunakan untuk menormalisasi semua fitur ke distribusi dengan mean=0 dan std=1.


In [ ]:
feature_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
                "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df_clean[feature_cols]), columns=feature_cols)
y = df_clean["Outcome"]

# Split data: 67% train, 33% test — stratified untuk menjaga proporsi kelas
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1/3, random_state=42, stratify=y
)

print(f"✅ Training set : {X_train.shape[0]} samples")
print(f"✅ Testing set  : {X_test.shape[0]} samples")
print(f"✅ Fitur        : {X_train.shape[1]} features")


---

## 6. Model Building & K Optimization

Melatih KNN dengan berbagai nilai K (1–14) untuk menemukan nilai K yang memberikan **test score tertinggi** sambil menghindari overfitting.


In [ ]:
train_scores, test_scores = [], []
k_range = range(1, 15)

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    test_scores.append(knn.score(X_test, y_test))

# Temukan K optimal
best_k = test_scores.index(max(test_scores)) + 1
print(f"🏆 K optimal    : k = {best_k}")
print(f"📈 Train score  : {max(train_scores)*100:.2f}%")
print(f"📊 Test score   : {max(test_scores)*100:.2f}%")


In [ ]:
# Visualisasi Train vs Test Score
plt.figure(figsize=(10, 5))
plt.plot(k_range, train_scores, marker="*", linewidth=2, label="Train Score", color="#2196F3")
plt.plot(k_range, test_scores, marker="o", linewidth=2, label="Test Score", color="#F44336")
plt.axvline(x=best_k, linestyle="--", color="gray", alpha=0.7, label=f"Best K = {best_k}")
plt.xlabel("Number of Neighbors (K)", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.title("KNN: Train vs Test Score untuk Setiap Nilai K", fontsize=14, fontweight="bold")
plt.legend(fontsize=11)
plt.xticks(k_range)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


---

## 7. Model Evaluation

Melatih model final dengan K optimal dan mengevaluasi performanya menggunakan **Confusion Matrix** dan **Classification Report**.

In [ ]:
# Model final
knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train, y_train)
y_pred = knn_final.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Diabetes", "Diabetes"],
            yticklabels=["Non-Diabetes", "Diabetes"],
            linewidths=1, linecolor="gray")
plt.title(f"Confusion Matrix (K={best_k})", fontsize=13, fontweight="bold")
plt.ylabel("Aktual", fontsize=11)
plt.xlabel("Prediksi", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Classification Report
acc = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy Model KNN (K={best_k}): {acc*100:.2f}%")
print("
📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Non-Diabetes", "Diabetes"]))


---

## 8. Conclusion

### 📌 Ringkasan

| Tahap | Keterangan |
|---|---|
| Dataset | 768 sampel, 8 fitur, 1 target biner |
| Missing Values | Ditemukan pada 5 kolom, ditangani dengan mean imputation |
| Scaling | StandardScaler untuk normalisasi fitur |
| Split Data | 67% train / 33% test (stratified) |
| K Optimal | K = 7 |
| Akurasi | ~80% |

### 🔍 Insight Utama

- **Glucose** memiliki korelasi tertinggi dengan outcome diabetes — fitur paling informatif dalam dataset ini.
- **Imputasi mean** dipilih untuk menangani missing values agar distribusi data tidak terdistorsi secara signifikan.
- Model KNN dengan **K=7** memberikan keseimbangan terbaik antara bias dan varians.
- Dataset ini memiliki **class imbalance** (65% non-diabetes vs 35% diabetes) yang dapat memengaruhi performa recall untuk kelas minoritas.

### 🚀 Potential Improvements

- Coba algoritma lain: Random Forest, SVM, atau XGBoost untuk perbandingan
- Tangani class imbalance menggunakan SMOTE atau 
- Lakukan hyperparameter tuning dengan GridSearchCV
- Gunakan cross-validation (k-fold) untuk evaluasi yang lebih robust
